# Structured PDF Extraction plus OCR (Gemini) plus GoogleTranslate


# SAMPLES AND TOKEN COUNTER

In [5]:
### SAMPLE DOCUMENTS
!wget -q -O handwriting_form.pdf https://storage.googleapis.com/generativeai-downloads/data/pdf_structured_outputs/handwriting_form.pdf
!wget -q -O invoice.pdf https://storage.googleapis.com/generativeai-downloads/data/pdf_structured_outputs/invoice.pdf
tender = "/content/drive/MyDrive/CRYSTAL STUFF/testset/IOC.pdf"
businesscard = "/content/drive/MyDrive/CRYSTAL STUFF/testset/businesscard.jpg"
gujarati = "/content/drive/MyDrive/CRYSTAL STUFF/testset/guj1.jpeg"
hindi = "/content/drive/MyDrive/CRYSTAL STUFF/testset/hindi1.jpg"
invoice = "/content/drive/MyDrive/CRYSTAL STUFF/testset/invoice.pdf"
handwriting_form = "/content/drive/MyDrive/CRYSTAL STUFF/testset/handwriting_form.pdf"

In [6]:
# Upload a sample PDF file and count tokens
import os

filename = tender
# Ensure the file exists
if os.path.exists(filename):
    sample_pdf = client.files.upload(file=filename, config={'display_name': 'sample'})
    file_size = client.models.count_tokens(model=model_id, contents=sample_pdf)
    print(f'The sample PDF file: {sample_pdf.display_name} is equal to {file_size.total_tokens} tokens')
else:
    print("Error: invoice.pdf not found. Please ensure the file is in the correct directory.")

Error: invoice.pdf not found. Please ensure the file is in the correct directory.


# CONFIGURATON

### Define google client calling, Pydantic classes and extraction handling

In [7]:
%%capture
%pip install "google-genai>=1" googletrans deep-translator

In [9]:
import tempfile
import os
import pandas as pd
from tabulate import tabulate
import textwrap
from google.colab import drive
from IPython.display import display # Import display
drive.mount('/content/drive')

#================ Define google client and LLM call #===========================
from google import genai
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
model_id = "gemini-2.0-flash-lite"


#============= Import necessary Pydantic models #===============================
from pydantic import BaseModel, Field
from typing import List, Optional

class Item(BaseModel):
    description: str = Field(description="The description of the item")
    quantity: float = Field(description="The Qty of the item")
    gross_worth: float = Field(description="The gross worth of the item")

class Invoice(BaseModel):
    """Extract the invoice number, date and all list items with description, quantity and gross worth and the total gross worth."""
    invoice_number: str = Field(description="The invoice number e.g. 1234567890")
    date: str = Field(description="The date of the invoice e.g. 2024-01-01")
    items: list[Item] = Field(description="The list of items with description, quantity and gross worth")
    total_gross_worth: float = Field(description="The total gross worth of the invoice")


class Form(BaseModel):
    """Extract the form number, fiscal start date, fiscal end date, and the plan liabilities beginning of the year and end of the year."""
    form_number: str = Field(description="The Form Number")
    start_date: str = Field(description="Effective Date")
    beginning_of_year: float = Field(description="The plan liabilities beginning of the year")
    end_of_year: float = Field(description="The plan liabilities end of the year")


class BusinessCard(BaseModel):
    """Extract information from a business card."""
    name: str = Field(description="The full name on the business card")
    title: str = Field(description="The job title on the business card")
    company: str = Field(description="The company name on the business card")
    phone: str = Field(description="The phone number on the business card")
    email: str = Field(description="The email address on the business card")
    website: str = Field(description="The website on the business card")
    address: str = Field(description="The address on the business card")


class TenderDocument(BaseModel):
    """Extract key information from a tender document."""
    tender_title: Optional[str] = Field(None, description="The title of the tender document.")
    tender_number: Optional[str] = Field(None, description="The unique identification number for the tender.")
    issuing_organization: Optional[str] = Field(None, description="The name of the organization issuing the tender.")
    submission_deadline: Optional[str] = Field(None, description="The deadline for submitting the tender proposal.")
    scope_of_work: Optional[str] = Field(None, description="A summary or description of the work to be performed.")
    scope_of_products_to_be_provided: Optional[str] = Field(None, description="A list of products or services to be provided.")
    eligibility_criteria: Optional[str] = Field(None, description="The requirements or qualifications bidders must meet.")
    submission_instructions: Optional[str] = Field(None, description="Instructions on how to prepare and submit the proposal.")
    general_terms_and_conditions_summary: Optional[str] = Field(None, description="summary of key general terms and conditions")
    specific_terms_and_conditions_summary: Optional[str] = Field(None, description="summary of key specific terms and conditions")
    contact_person: Optional[str] = Field(None, description="The name of the contact person for inquiries.")
    contact_information: Optional[str] = Field(None, description="Contact details (phone, email, address) for inquiries.")
    evaluation_criteria: Optional[str] = Field(None, description="The criteria used to evaluate the submitted proposals.")
    required_documents: Optional[List[str]] = Field(None, description="A list of documents that must be included in the submission.")
    site_visit_details: Optional[str] = Field(None, description="Information about any mandatory or optional site visits.")
    earnest_money_deposit_terms: Optional[str] = Field(None, description="Details regarding the earnest money deposit.")
    emd_terms: Optional[str] = Field(None, description="Details regarding the terms of the earnest money deposit.")
    payment_terms: Optional[str] = Field(None, description="Details regarding payment schedules and terms.")
    contract_duration: Optional[str] = Field(None, description="The expected duration of the contract.")


# Define Topic before Person
class Topic(BaseModel):
    name: str = Field(description="The name of the topic")


class Person(BaseModel):
    first_name: str = Field(description="The first name of the person")
    last_name: str = Field(description="The last name of the person")
    age: int = Field(description="The age of the person, if not provided please return 0")
    work_topics: list[Topic] = Field(description="The fields of interest of the person, if not provided please return an empty list")


#============= Define Structured Output requirement to the LLM #================
def extract_structured_data(file_path: str, model: BaseModel):
    # Upload the file to the File API
    file = client.files.upload(file=file_path, config={'display_name': file_path.split('/')[-1].split('.')[0]})
    # Generate a structured response using the Gemini API
    prompt = f"Extract the structured data from the following PDF file"
    response = client.models.generate_content(model=model_id, contents=[prompt, file], config={'response_mime_type': 'application/json', 'response_schema': model})
    # Convert the response to the pydantic model and return it
    return response.parsed

# Define a basic text wrapping function
def wrap_text(text, width):
    if isinstance(text, str):
        return '\n'.join(textwrap.wrap(text, width=width))
    return text


def handle_extraction(change):
    # Clear previous output
    output_widget.clear_output()

    with output_widget:
        if not upload_widget.value:
            print("Please upload a document first.")
            return

        uploaded_file_info = upload_widget.value

        # Assuming single file upload, get the first (and only) file's data
        file_name = list(uploaded_file_info.keys())[0]
        file_content = uploaded_file_info[file_name]['content']
        file_extension = os.path.splitext(file_name)[1].lower()

        # Define allowed file extensions (you can customize this list)
        allowed_extensions = ['.pdf', '.jpg', '.jpeg', '.png']

        if file_extension not in allowed_extensions:
            print(f"Error: Invalid file type '{file_extension}'. Please upload a file with one of the allowed extensions: {', '.join(allowed_extensions)}")
            return


        selected_type = dropdown_widget.value

        if selected_type == 'Translate Text':
            # Handle text extraction and translation
            temp_file_path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=file_extension) as temp_file_handle:
                    temp_file_handle.write(file_content)
                    temp_file_path = temp_file_handle.name

                print(f"Extracting and translating text from {file_name}...")

                # Call the text extraction and translation function
                extracted_text = extract_text_translate(temp_file_path)

                if extracted_text:
                    print("\n--- Extracted and Translated Text ---")
                    print(extracted_text)
                else:
                    print("No text extracted or translated.")

            except Exception as e:
                print(f"An error occurred during text extraction and translation: {e}")
            finally:
                # Clean up the temporary file
                if temp_file_path and os.path.exists(temp_file_path):
                    os.remove(temp_file_path)

        else:
            # Map selected document type to Pydantic model for structured extraction
            model_mapping = {
                'Invoice': Invoice,
                'Form': Form,
                'BusinessCard': BusinessCard,
                'TenderDocument': TenderDocument,
                'Person': Person
            }

            selected_model = model_mapping.get(selected_type)

            if not selected_model:
                print(f"Error: Invalid document type selected: {selected_type}")
                return

            # Save the uploaded file content temporarily
            temp_file_path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=file_extension) as temp_file_handle:
                    temp_file_handle.write(file_content)
                    temp_file_path = temp_file_handle.name

                print(f"Extracting data using {selected_type} model from {file_name}...")

                # Call the extraction function
                extracted_data = extract_structured_data(temp_file_path, selected_model)

                print("\n--- Extracted Data ---")
                # Display the extracted data in a tabular format with wrapped text
                if extracted_data:
                    data_for_table = []
                    # Define desired wrap width
                    wrap_width = 80 # You can adjust this width as needed

                    if selected_type == 'Invoice':
                        # Special handling for Invoice to display items
                        data_for_table.append(["Invoice Number", extracted_data.invoice_number])
                        data_for_table.append(["Date", extracted_data.date])
                        data_for_table.append(["Total Gross Worth", extracted_data.total_gross_worth])
                        data_for_table.append(["--- Items ---", ""])
                        for item in extracted_data.items:
                             data_for_table.append(["Description", wrap_text(item.description, wrap_width)])
                             data_for_table.append(["Quantity", item.quantity])
                             data_for_table.append(["Gross Worth", item.gross_worth])
                             data_for_table.append(["-------------", ""]) # Separator for items

                    elif selected_type == 'Person':
                         # Special handling for Person to display work topics
                         data_for_table.append(["First Name", extracted_data.first_name])
                         data_for_table.append(["Last Name", extracted_data.last_name])
                         data_for_table.append(["Age", extracted_data.age])
                         data_for_table.append(["--- Work Topics ---", ""])
                         if extracted_data.work_topics:
                             for topic in extracted_data.work_topics:
                                 data_for_table.append(["Topic", wrap_text(topic.name, wrap_width)])
                             data_for_table.append(["-------------", ""]) # Separator for topics
                         else:
                             data_for_table.append(["Topic", "No work topics found."])


                    else:
                        # General handling for other models
                        for field, value in extracted_data.model_dump().items():
                            # Apply text wrapping
                            wrapped_value = wrap_text(value, wrap_width)
                            data_for_table.append([field.replace('_', ' ').title(), wrapped_value])

                    table_output = tabulate(data_for_table, headers=["Field", "Value"], tablefmt="grid")
                    print(table_output) # Use display instead of print
                else:
                    print("No data extracted.")


            except Exception as e:
                print(f"An error occurred during extraction: {e}")
            finally:
                # Clean up the temporary file
                if temp_file_path and os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
                    #print(f"Temporary file {temp_file_path} removed.") # Optional: for debugging


#============= Translation Module #================
from deep_translator import GoogleTranslator

def extract_text_translate (file_path: str):
    try:
        # Upload the file to the File API
        file = client.files.upload(file=file_path, config={'display_name': file_path.split('/')[-1].split('.')[0]})
        file_size = client.models.count_tokens(model=model_id,contents=file)
        print(f'File: {file.display_name} equals to {file_size.total_tokens} tokens')

        # Use a model capable of handling images/PDFs for text extraction
        # Ensure this model is suitable for multimodal input if handling images/PDFs
        extraction_model_id = "gemini-2.0-flash-lite" # Or another suitable multimodal model

        # Define a simple prompt to extract text
        prompt = "Extract all the text from this document."

        # Generate content with the file
        response = client.models.generate_content(model=extraction_model_id, contents=[prompt, file])

        # Get the extracted text
        extracted_text = response.text
        print(f"Extracted Text: {extracted_text}")

        # Translate the extracted text
        translator = GoogleTranslator(source='auto', target='en') # auto-detect source language, translate to English
        translated_text = translator.translate(extracted_text)

        return translated_text

    except Exception as e:
        print(f"An error occurred during text extraction or translation: {e}")
        return None

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# UI Module

### Document uploading, Type selction and Extraction

In [10]:
from ipywidgets import FileUpload, Dropdown, Button, VBox, Output
from IPython.display import display

# Define basemodel names for user to select
model_names = ['Invoice', 'Form', 'BusinessCard', 'TenderDocument', 'Person', 'Translate Text']
dropdown_widget = Dropdown(
    options=model_names, # pydantic Basemodel names
    description='Select Document Type:'
)

# Create an exract button to display results
extract_button = Button(description='Extract Data')

# Link the button's click event to the handler function
extract_button.on_click(handle_extraction)

# Create an upload widget to display results
upload_widget = FileUpload(
    accept='',  # Accept all file types
    multiple=False,  # Allow only one file to be uploaded
    description='Upload Document'
)

# Create an output widget to display results
output_widget = Output()

#=======Arrange widgets vertically using VBox with spacing and styling==========
ui = VBox([upload_widget, dropdown_widget, extract_button, output_widget],
          layout={'border': '1px solid lightgray', 'padding': '20px', 'margin': '20px'}, # Add border, padding, and margin to the container
          )

# Add some styling to individual widgets
upload_widget.layout.margin = '0 0 20px 0' # Add margin below the upload widget
dropdown_widget.layout.margin = '0 0 20px 0' # Add margin below the dropdown widget
extract_button.style.button_color = 'lightblue' # Change button color
extract_button.layout.margin = '0 0 20px 0' # Add margin below the button

# Execution

In [11]:
# Display the designed UI
display(ui)